# 第 28 章 · Prompt 缓存：先判断证据，再计算收益

相同请求更快，并不能单独证明命中缓存。首先确认 usage 的字段和语义，再分开计算 token 缓存占比、请求命中占比、端到端耗时和任务总成本。

本章运行离线统计与配对 bootstrap。它不实现 provider 缓存、不发真实请求，也不拿合成延迟证明性能提升。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · 分母只含有资格的观测

input 和 cached 必须同时已知，cached 不能大于 input，bool 也不能当 token 数。0/0 的比例应是 None。调整 `cached_tokens` 到 101，观察异常观测被排除及理由。

In [2]:
m = load_example('28', 'cache_stats')
def attempt(key, input_tokens, cached=None, *, status='completed', phase='measure'):
    usage = {'input_tokens': input_tokens}
    if cached is not None:
        usage['input_tokens_details'] = {'cached_tokens': cached}
    return m.Attempt(key, 'task-' + key, phase, status, usage)
rows = [attempt('a', 100, 80), attempt('b', 300, 0), attempt('unknown', 1000)]
summary = m.summarize(rows)
assert summary['eligible_input_tokens'] == 400
assert summary['cached_token_fraction'] == 0.2
assert summary['positive_cached_attempt_fraction'] == 0.5
assert summary['usage_coverage'] == 2/3
assert summary['known_input_tokens_all_attempts'] == 1400
assert m.qualify(attempt('invalid', 100, 101))[1] == 'cached_exceeds_input'
assert m.summarize([attempt('zero', 0, 0)])['cached_token_fraction'] is None
print(summary)

{'evidence_kind': 'fixture', 'attempts': 3, 'duplicate_records': 0, 'phase_counts': {'measure': 3}, 'status_counts': {'completed': 3}, 'eligible_attempts': 2, 'usage_coverage': 0.6666666666666666, 'eligible_input_tokens': 400, 'eligible_cached_tokens': 80, 'cached_token_fraction': 0.2, 'positive_cached_attempt_fraction': 0.5, 'excluded': {'unknown': 'cached_details_missing'}, 'known_input_tokens_all_attempts': 1400, 'known_input_coverage': 1.0, 'total_cost': None}


## 2 · 预热、失败和取消不能从成本集合消失

重复记录去重，实际重试保留。取消请求如果带已知用量，同样进入 token 分子分母；没有用量则保留为排除项。phase 是实验条件，不是免单依据。

In [3]:
from dataclasses import replace
warmup = attempt('warmup', 200, 0, phase='prewarm')
failed = attempt('failed', 100, 50, status='failed')
cancelled = m.Attempt('cancelled', 'task-cancelled', 'measure', 'cancelled', None)
retry = replace(rows[0], attempt_id='real-retry')
cohort = m.summarize([*rows, rows[0], warmup, failed, cancelled, retry])
assert cohort['attempts'] == 7 and cohort['duplicate_records'] == 1
assert cohort['phase_counts']['prewarm'] == 1
assert cohort['status_counts']['cancelled'] == 1 and cohort['status_counts']['failed'] == 1
assert cohort['eligible_input_tokens'] == 800
assert cohort['eligible_cached_tokens'] == 210
assert cohort['excluded']['cancelled'] == 'raw_usage_unavailable'
assert cohort['total_cost'] is None
print({k: cohort[k] for k in ('attempts', 'phase_counts', 'status_counts', 'cached_token_fraction', 'excluded')})

{'attempts': 7, 'phase_counts': {'measure': 6, 'prewarm': 1}, 'status_counts': {'completed': 5, 'failed': 1, 'cancelled': 1}, 'cached_token_fraction': 0.2625, 'excluded': {'unknown': 'cached_details_missing', 'cancelled': 'raw_usage_unavailable'}}


## 3 · 统计入口拒绝冲突和 fixture/live 混合

恢复时同一 attempt 的 usage 改了，不能覆盖旧证据再得到更漂亮的平均数。真实观测与合成观测需要不同 cohort；下方故意写 live 标签只是反例，不构成真实服务证据。

In [4]:
for values, fragment in [([rows[0], replace(rows[0], status='failed')], 'conflicting attempt'),
                          ([rows[0], replace(rows[1], evidence_kind='live')], 'cannot share a cohort')]:
    try:
        m.summarize(values)
    except ValueError as error:
        assert fragment in str(error)
    else:
        raise AssertionError(fragment)
restored = m.summarize([*rows, *rows])
assert restored['attempts'] == 3 and restored['duplicate_records'] == 3
assert restored['cached_token_fraction'] == summary['cached_token_fraction']
print({'restore_duplicate_records': restored['duplicate_records'], 'ratio_preserved': restored['cached_token_fraction']})

{'restore_duplicate_records': 3, 'ratio_preserved': 0.2}


## 4 · 任务级配对，避免把重试当独立样本

每个 pair 是一个独立任务或实验 block，值应是该任务含重试的总指标。bootstrap 对配对差值重采样。下面的合成毫秒数仅测试算法；负区间也不能证明真实 provider 更快。把 task ID 改为重复值，入口应拒绝伪重复样本。

In [5]:
pairs = [('t1', 120, 100), ('t2', 150, 140), ('t3', 90, 85), ('t4', 200, 170)]
interval = m.paired_mean_interval(pairs, seed=28, resamples=1000)
assert interval['mean_treatment_minus_baseline'] == -16.25
assert interval['percentile_ci95'][0] <= -16.25 <= interval['percentile_ci95'][1]
try:
    m.paired_mean_interval([pairs[0], pairs[0]], resamples=100)
except ValueError as error:
    assert 'pseudoreplication' in str(error)
else:
    raise AssertionError('重复任务不能当独立 pair')
print({'synthetic_metric': 'task_total_ms', **interval, 'live_speedup_claim': False})

{'synthetic_metric': 'task_total_ms', 'pairs': 4, 'mean_treatment_minus_baseline': -16.25, 'percentile_ci95': [-27.5, -6.25], 'seed': 28, 'resamples': 1000, 'method': 'paired_percentile_bootstrap', 'live_speedup_claim': False}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/core/src/client.rs#L1190-L1224](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/client.rs#L1190-L1224)
- [codex-rs/core/src/client_tests.rs#L1139-L1220](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/client_tests.rs#L1139-L1220)
- [codex-rs/core/src/client.rs#L711-L742](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/client.rs#L711-L742)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。